<a href="https://colab.research.google.com/github/Youseef3550/flyrank-ml-yousef-assighn-1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two signal checks — do the signals my rule leans on actually hold?

**Signals I'm checking, and why these two:**

1. **Staleness → `days_since_last_update`.** This is the signal behind FlyRank's
   `stale_visible_page` refresh flag (the session's rule: stale AND visible = worth a look).
   The naive story: "the longer since a page was touched, the more likely it's declining."
2. **CTR-vs-position → `ctr` read against `position_tier`.** This is the signal behind the
   session's `low_ctr_visible_page` / CTR-fix logic: a page ranking well but getting fewer
   clicks than its position deserves is a fixable, high-leverage problem.

Both are flag-linked (satisfies the "at least one" requirement twice over), because these are
the two beliefs my rule below actually leans its weight on. `is_declining_label` is only used
here to **audit** the signals against a real outcome — it never becomes a score input.

In [1]:
import numpy as np
import pandas as pd
import subprocess
from pathlib import Path

candidate_paths = [
    Path("../../data/raw/content_refresh_anonymized.csv"),  # run from work/notebooks/ inside a cloned repo
    Path("data/raw/content_refresh_anonymized.csv"),         # run from repo root
]
RAW_PATH = next((p for p in candidate_paths if p.exists()), None)

if RAW_PATH is None:
    # Standalone Colab session (notebook opened directly, repo not cloned) -- clone it once.
    repo_dir = Path("flyrank-ml-yousef-assighn-1")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Youseef3550/flyrank-ml-yousef-assighn-1.git", str(repo_dir)],
            check=True,
        )
    RAW_PATH = repo_dir / "data" / "raw" / "content_refresh_anonymized.csv"

print("Using data file:", RAW_PATH.resolve())
df = pd.read_csv(RAW_PATH)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df.shape)
print("Overall decline rate (base rate):", round(df["is_declining_label"].mean(), 3))

MIN_N = 50  # sample-size floor from the auditing-signals skill

# --- Signal check A: staleness vs decline rate ---------------------------------
print("\n=== Signal A: staleness (freshness_tier) vs decline rate ===")
tier_order = ["0-30", "31-90", "91-180", "181+"]
signal_a = (
    df.groupby("freshness_tier", observed=True)
    .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
    .reindex(tier_order)
)
signal_a["decline_rate"] = signal_a["decline_rate"].round(3)
print(signal_a)
below_floor_a = signal_a[signal_a["n"] < MIN_N]
if len(below_floor_a):
    print(f"\nBuckets below the n={MIN_N} floor (read with caution):\n{below_floor_a}")

# --- Signal check B: CTR vs position ---------------------------------------------
print("\n=== Signal B: CTR vs position (position_tier) ===")
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
signal_b = (
    df[df["position_tier"] != "no_data"]
    .groupby("position_tier", observed=True)
    .agg(n=("content_id", "size"), median_ctr=("ctr", "median"), mean_ctr=("ctr", "mean"),
         decline_rate=("is_declining_label", "mean"))
    .reindex(pos_order)
)
signal_b[["median_ctr", "mean_ctr", "decline_rate"]] = signal_b[["median_ctr", "mean_ctr", "decline_rate"]].round(3)
print(signal_b)
below_floor_b = signal_b[signal_b["n"] < MIN_N]
if len(below_floor_b):
    print(f"\nBuckets below the n={MIN_N} floor (read with caution):\n{below_floor_b}")

Using data file: /content/flyrank-ml-yousef-assighn-1/data/raw/content_refresh_anonymized.csv
(30000, 45)
Overall decline rate (base rate): 0.542

=== Signal A: staleness (freshness_tier) vs decline rate ===
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471

=== Signal B: CTR vs position (position_tier) ===
                   n  median_ctr  mean_ctr  decline_rate
position_tier                                           
top_3           2321        0.00     1.484         0.241
page_1         11814        0.16     0.652         0.570
striking        7304        0.11     0.323         0.610
page_3_5        7242        0.03     0.222         0.562
deep            1319        0.00     0.150         0.344


**Verdict A — staleness → decline rate: `MIXED`.**
`decline_rate` does NOT climb steadily with `freshness_tier`: it goes 0.51 (0-30 days) →
0.59 (31-90) → 0.61 (91-180) → then **drops back to 0.47** at 181+, the tier that should be
the most "obviously stale." All four buckets clear the n=50 floor (smallest is 174), so this
isn't noise from a tiny cell — the pattern genuinely reverses. **What this means in practice:**
staleness alone is not a clean decline predictor on this data; it can contribute to a score
(older content is still a fair signal to combine with others) but it must never carry the
rule by itself, and a "stale = declining" one-line story would be wrong. Calling this MIXED
instead of forcing it to CONFIRMED already saved the rule from over-weighting staleness.

**Verdict B — CTR falls as position gets worse: `CONFIRMED`.**
Mean CTR drops monotonically and by a wide margin as position gets worse: top_3 ≈ 1.48,
page_1 ≈ 0.65, striking ≈ 0.32, page_3_5 ≈ 0.22, deep ≈ 0.15 (remember, these are ×100
percentages — sub-2% CTRs throughout). n ranges from ~1.3k to ~11.8k per bucket, comfortably
above the floor. `median_ctr` looks noisier (it's 0.00 for `top_3` — a heavy-tailed rate
metric where the median sits at zero even though the mean is high, exactly the auditing-signals
warning about heavy tails), which is why I lean on the mean and the monotonic direction here,
not the median alone. **What this means in practice:** "low CTR for the position a page holds"
is a real, checkable signal — the CTR-fix belief holds up, and it earns real weight below.

## 2. Build the ranked queue (writes the CSV)

**The rule, in plain words:** A page is worth reviewing first if it already pulls real search
visibility (it has an audience worth fixing for), its click-through rate is worse than other
pages sitting at the same position (a fixable CTR gap — this is the CONFIRMED signal above,
so it gets the most weight after visibility), and it hasn't been touched in a while (the
MIXED staleness signal — kept, but at low weight, since the audit showed it doesn't move
cleanly with decline on its own).

**Score** (no fitted weights, just three readable pieces):

```
visibility_score = percentile_rank(log1p(impressions_90d))
ctr_gap_score     = 1 − percentile_rank(ctr WITHIN its own position_tier)   # "low ctr for ITS position"
staleness_score   = percentile_rank(days_since_last_update)

baseline_action_score = 0.45*visibility_score + 0.35*ctr_gap_score + 0.20*staleness_score
```

**ONE reason code per row** (priority order, mutually exclusive — not a joined list):
1. `low_ctr_for_position` — visible (`impressions_90d ≥ 500`), ranks 1-20, and sits in the
   bottom quartile of CTR for its own position tier. This is the session's `low_ctr_visible_page`
   flag, re-derived from the CONFIRMED signal above.
2. `stale_visible_page` — visible (`impressions_90d ≥ 500`) and untouched ≥ 180 days. Same
   flag the session used, kept at low confidence given the MIXED audit above.
3. `general_review` — everything else; still ranked, just with no specific reason to point to.

**Action label**, one-to-one from the reason code:
`low_ctr_for_position → refresh_and_review_ctr`, `stale_visible_page → refresh`,
`general_review → monitor`.

**Inputs used:** `impressions_90d`, `ctr`, `position_tier`/`avg_position`,
`days_since_last_update`. All are trailing-90-day, same-window observed signals — no
`trend_pct`/`trend_direction` (the label source), no future window, no FlyRank product flag
(`health_score`, `priority_score`, `is_quick_win`, `action_type` don't exist in this dataset at all).

In [2]:
def ctr_reason_code(row):
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr_pct_within_position"] <= 0.25:
        return "low_ctr_for_position"
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page"
    return "general_review"


ACTION_BY_REASON = {
    "low_ctr_for_position": "refresh_and_review_ctr",
    "stale_visible_page": "refresh",
    "general_review": "monitor",
}


def percentile_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method="average")


baseline = df.copy()

baseline["visibility_score"] = percentile_rank(np.log1p(baseline["impressions_90d"]))
baseline["staleness_score"] = percentile_rank(baseline["days_since_last_update"])

# CTR read AGAINST its own position tier -- "low CTR for where it already ranks"
baseline["ctr_pct_within_position"] = baseline.groupby("position_tier")["ctr"].rank(pct=True)
baseline["ctr_gap_score"] = 1 - baseline["ctr_pct_within_position"].fillna(0.5)

baseline["baseline_action_score"] = (
    0.45 * baseline["visibility_score"]
    + 0.35 * baseline["ctr_gap_score"]
    + 0.20 * baseline["staleness_score"]
).clip(0, 1)

baseline["reason_code"] = baseline.apply(ctr_reason_code, axis=1)
baseline["action"] = baseline["reason_code"].map(ACTION_BY_REASON)
baseline["rank"] = baseline["baseline_action_score"].rank(method="first", ascending=False).astype(int)

baseline = baseline.sort_values("rank")

print("Reason code counts:")
print(baseline["reason_code"].value_counts())
print("\nAction counts:")
print(baseline["action"].value_counts())

output_columns = [
    "rank", "content_id", "client_id", "baseline_action_score",
    "visibility_score", "ctr_gap_score", "staleness_score",
    "reason_code", "action",
    "impressions_90d", "avg_position", "ctr", "days_since_last_update",
    "word_count", "trend_direction", "is_declining_label",
]

if (Path("../../data")).exists():
    out_path = Path("../outputs/baseline_action_score.csv")       # inside cloned repo, work/notebooks/
elif Path("data").exists():
    out_path = Path("work/outputs/baseline_action_score.csv")     # run from repo root
else:
    out_path = Path("flyrank-ml-yousef-assighn-1/work/outputs/baseline_action_score.csv")  # standalone Colab clone
out_path.parent.mkdir(parents=True, exist_ok=True)
baseline[output_columns].to_csv(out_path, index=False)
print(f"\nWrote ranked queue: {out_path.resolve()}")
print(f"Rows written: {len(baseline)}")

Reason code counts:
reason_code
general_review          28822
low_ctr_for_position     1161
stale_visible_page         17
Name: count, dtype: int64

Action counts:
action
monitor                   28822
refresh_and_review_ctr     1161
refresh                      17
Name: count, dtype: int64

Wrote ranked queue: /content/flyrank-ml-yousef-assighn-1/work/outputs/baseline_action_score.csv
Rows written: 30000


## 3. Top-10 review

*The task card asks for a top-10 hand review (the `building-baselines` skill's own default
habit is top-20 — ten is the floor this card asks for, twenty is always welcome; I did ten to
match the card exactly). One line each: the action, why it's there, what would make it wrong.

In [3]:
top10 = baseline.head(10)[[
    "rank", "content_id", "action", "reason_code", "baseline_action_score",
    "impressions_90d", "avg_position", "ctr", "days_since_last_update", "trend_direction",
]]
print(top10.to_string(index=False))

 rank           content_id                 action          reason_code  baseline_action_score  impressions_90d  avg_position  ctr  days_since_last_update trend_direction
    1 content_c8e9d6ab9013 refresh_and_review_ctr low_ctr_for_position               0.908391           208678           9.7 0.00                     104            down
    2 content_fb4bf6555c79                monitor       general_review               0.882417            84093          45.6 0.00                     104            down
    3 content_825a9788af8d refresh_and_review_ctr low_ctr_for_position               0.876073            16786           5.6 0.00                     104            down
    4 content_6e28a04c07a8                monitor       general_review               0.875742            41226          34.3 0.00                     104            down
    5 content_8ba781dafa55 refresh_and_review_ctr low_ctr_for_position               0.874731            16156           9.0 0.00                     

**Top-10, one line each (action — why — what would make it wrong):**

1. **`refresh_and_review_ctr`** — 208,678 impressions at position 9.7 with a 0.00% CTR is the
   single clearest CTR-vs-position gap in the whole set. *Wrong if:* that 0.00% CTR is a
   tracking artifact (e.g. a redirect, or a page GSC can't attribute clicks to) rather than a
   real reader problem — worth a manual SERP check before assigning an editor.
2. **`monitor`** — 84,093 impressions but position 45.6 puts it on page 5, well outside the
   `≤20` band the CTR reason code requires, so it falls through to `general_review` purely on
   visibility plus a rock-bottom CTR-gap score. *Wrong if:* a page this deep can't realistically
   be pulled into the top 20 by a refresh — the score is rewarding raw impression volume, not an
   achievable win, which is this rule's biggest honest weakness (see Section 4).
3. **`refresh_and_review_ctr`** — 16,786 impressions at position 5.6, near-zero CTR: strong
   position, weak clicks, a textbook fixable gap. *Wrong if:* the title/meta already matches
   best practice and the low CTR is really a low-intent query mismatch no refresh fixes.
4. **`monitor`** — same pattern as #2: high impressions (41,226), deep position (34.3), CTR
   gap doing all the lifting. *Wrong if:* a deep-position, high-impression page like this turns
   out to be a seasonal spike rather than a stable audience worth refreshing for.
5. **`refresh_and_review_ctr`** — 16,156 impressions at position 9.0, 0.00% CTR, one of the
   cleanest "ranks well, gets no clicks" cases in the queue. *Wrong if:* `impressions_90d` is
   inflated by a single traffic spike rather than steady visibility (worth checking
   `days_with_impressions` before committing an editor's time).
6. **`monitor`** — 32,491 impressions, position 36.2, `trend_direction = stable` (not even
   declining). *Wrong if:* "stable" here means it already recovered on its own — refreshing a
   page that isn't actually hurting wastes review capacity the rule is supposed to protect.
7. **`monitor`** — 30,962 impressions, position 30.7, 0.00% CTR, declining. Same deep-position
   pattern as #2/#4/#6. *Wrong if:* the page targets a keyword where even position 1 wouldn't
   earn many clicks (a low-intent informational query), making the "gap" look bigger than it is.
8. **`monitor`** — 25,748 impressions, position 48.5 — the deepest position in the top 10.
   *Wrong if:* this is scored on visibility alone with almost no realistic path to page 1;
   the highest-risk "score inflated by volume" case in this list.
9. **`monitor`** — 23,513 impressions, position 39.4, declining. *Wrong if:* client-level
   context (a brand-new content type, or a client that never ranks past position 30) makes this
   normal for that page rather than a red flag.
10. **`refresh_and_review_ctr`** — 79,035 impressions at position 8.7 with a small but real
    0.07% CTR (not zero), `trend_direction = stable`. *Wrong if:* "stable" plus a non-zero,
    if-small CTR means this page is already near its ceiling for its query, and a refresh
    spends effort where there's little headroom left.

## 4. Weak picks + leakage check

*Which picks look wrong, and why? Confirm no product flags or future windows leaked in.*

**Weak picks, named honestly:** five of the top 10 (#2, #4, #6, #7, #8) carry the
`general_review` → `monitor` label, not because they matched a specific reason code, but
because raw visibility (huge impressions) plus a rock-bottom CTR-gap score pushed them to the
top of a **linear** score even at position 30-48 — well outside any realistic "refresh gets it
back to page 1" story. That's the honest weakness the top-10 review surfaced: the rule can't
express "this gap is unfixable because the position is too deep," it can only add three
percentile scores. A model can learn that interaction (visibility × position × CTR gap,
non-additively); the fixed rule can't — and that gap is exactly this baseline's job: to be
honestly beatable.

**Leakage / product-flag check, run against the actual columns used:**

In [4]:
SCORE_INPUTS = ["impressions_90d", "ctr", "position_tier", "avg_position", "days_since_last_update"]
FORBIDDEN_LABEL_COLS = ["trend_direction", "trend_pct", "is_declining_label"]
FORBIDDEN_PRODUCT_FLAGS = ["health_score", "priority_score", "is_quick_win", "needs_ctr_fix", "action_type"]

print("Score inputs actually used:", SCORE_INPUTS)
print("Any label-derived column among them?", any(c in SCORE_INPUTS for c in FORBIDDEN_LABEL_COLS))
print("Any FlyRank product flag present among the dataset's columns at all?",
      any(c in df.columns for c in FORBIDDEN_PRODUCT_FLAGS))
print("Any future-window column (last_30d/prev_30d) used in the score?",
      any("30d" in c for c in SCORE_INPUTS))

print("\nAll metrics used are trailing-90-day, same-snapshot signals -- ")
print("no forward-looking window exists in this starter CSV to leak from in the first place.")

Score inputs actually used: ['impressions_90d', 'ctr', 'position_tier', 'avg_position', 'days_since_last_update']
Any label-derived column among them? False
Any FlyRank product flag present among the dataset's columns at all? False
Any future-window column (last_30d/prev_30d) used in the score? False

All metrics used are trailing-90-day, same-snapshot signals -- 
no forward-looking window exists in this starter CSV to leak from in the first place.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.